# 2 · Phase 2 full raw YOLO11n baseline, then controlled benchmark

Default: all prepared raw train/tune data, **100 epochs maximum**, native early
stopping, per-epoch checkpoints and native resume. The raw baseline must complete
before preprocessing experiments. A single frozen raw checkpoint compares JPEG95,
Gamma, CLAHE, Denoise, Sharpen and Adaptive; recipes are calibrated on train/tune.

Evaluation defaults to **tune**. Official-val holdout stays unprepared and unevaluated.
Changing to final test requires an explicit frozen protocol and a separately prepared
holdout root. The notebook displays actual native metrics; missing results stay missing.


In [ ]:
import importlib.metadata as md
import io
import platform
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from IPython.display import Image, clear_output, display
from PIL import Image as PILImage
from ultralytics import YOLO
from ultralytics.utils.ops import xywh2xyxy

from vehicle_detection.common import (
    digest,
    file_hash,
    read_image,
    read_json,
    reuse_evaluation,
    write_json,
)
from vehicle_detection.data import CLASSES
from vehicle_detection.offline import verify_alignment

ROOT = Path.cwd() if (Path.cwd() / "pyproject.toml").exists() else Path.cwd().parent
MODE = "full"
RUN_ROOT = ROOT / "runs" / f"phase2-{MODE}"
TRAIN_VARIANTS = ["raw_original"]
EVAL_VARIANTS = ["raw_original"]
MODEL = "yolo11n.pt"
SEED, EPOCHS, BATCH, IMGSZ, DEVICE = 2026, 100, 32, 640, "0"
PATIENCE, WORKERS, SAVE_PERIOD, RESUME = 20, 8, 1, True
EVAL_SPLIT = "tune"
FINAL_PROTOCOL = None  # Path to a frozen protocol, required for final test release.
EXPERIMENT_ROOT = RUN_ROOT / "experiments" / "yolo11n-seed2026-v1"
FROZEN_FOR_TEST = False
VAL_SETTINGS = {
    "imgsz": IMGSZ,
    "batch": BATCH,
    "device": DEVICE,
    "conf": 0.001,
    "iou": 0.7,
    "max_det": 300,
    "quantize": 32,
    "half": False,
    "seed": SEED,
    "deterministic": True,
    "workers": 0,
    "plots": True,
    "verbose": False,
}
DEEP_VERIFY_EXPORTS = False  # Set True for on-demand all-image hashing/decoding.
FORCE_REEVALUATE = False
assert EVAL_SPLIT in {"tune", "test"}
if EVAL_SPLIT == "test":
    assert FROZEN_FOR_TEST and FINAL_PROTOCOL is not None and Path(FINAL_PROTOCOL).is_file()
assert TRAIN_VARIANTS[0] == "raw_original" and "raw_original" in EVAL_VARIANTS
VARIANTS = sorted(set(TRAIN_VARIANTS + EVAL_VARIANTS))
DATASETS = {v: RUN_ROOT / "datasets" / v for v in VARIANTS}
assert all((p / "data.yaml").is_file() for p in DATASETS.values()), "Run preparation notebook first"
print("Aligned exported images:", verify_alignment(DATASETS.values(), deep=DEEP_VERIFY_EXPORTS))
dataset_identities = {
    v: {
        "export": read_json(p / "export.json"),
        "files_identity": read_json(p / "verified.json")["files_identity"],
    }
    for v, p in DATASETS.items()
}
display(pd.DataFrame({v: d["export"]["encoding"] for v, d in dataset_identities.items()}).T)
evaluation_versions = {
    p: md.version(p) for p in ["ultralytics", "torch", "torchvision", "numpy", "opencv-python"]
}
evaluation_hardware = {
    "platform": platform.platform(),
    "processor": platform.processor(),
    "cuda_runtime": torch.version.cuda,
    "gpu": torch.cuda.get_device_name(int(DEVICE))
    if DEVICE.isdigit() and torch.cuda.is_available()
    else None,
}
selection = read_json(RUN_ROOT / "selection.json")
assert selection["splits"].get(EVAL_SPLIT), "Requested evaluation split has not been released"
print("Split sizes:", {s: len(v) for s, v in selection["splits"].items()})
print("Reserved holdout IDs (no pixel access):", len(selection.get("reserved_test_ids", [])))
print("Each data.yaml: train=images/train, val=images/tune, test=images/test")
display(
    pd.Series(
        {p: md.version(p) for p in ["ultralytics", "torch", "torchvision", "numpy"]},
        name="versions",
    )
)
plt.rcParams.update({"figure.dpi": 90, "font.size": 9})

## Independently train each dataset variant, with native checkpoints/resume

Every run starts from the **same pretrained file/hash**, has the same seed,
epochs/batch/resolution/device, and gets a distinct directory with best/last
checkpoints. Standard Ultralytics augmentation is comparable across variants.
An existing directory must match its saved configuration and export identity.
Completed runs verify best/last hashes and reuse them; interrupted runs use
`YOLO(last.pt).train(resume=True)`. No custom training/checkpoint engine is used.
Change `EXPERIMENT_ROOT` for different settings instead of mixing outputs.

`PATIENCE` controls native early stopping; `SAVE_PERIOD` controls periodic snapshots,
with native best/last always saved. `RESUME` controls interrupted-run recovery.
The default trains only raw, on all available train/tune examples. Native epoch=-1
after optimizer stripping is the completion signal, including early stopping.

In [ ]:
# Gate later treatments before any variant training can start.
if TRAIN_VARIANTS != ["raw_original"] or EVAL_VARIANTS != ["raw_original"]:
    assert (EXPERIMENT_ROOT / "raw_original/completed.json").is_file(), (
        "Finish full raw baseline first"
    )
initial = YOLO(MODEL)
pretrained = Path(initial.ckpt_path).resolve()
pretrained_sha256 = file_hash(pretrained)
print("Identical starting checkpoint:", pretrained.name, pretrained_sha256)
checkpoints, training_log = {}, []
for variant in TRAIN_VARIANTS:
    directory = EXPERIMENT_ROOT / variant
    identity = {
        "variant": variant,
        "seed": SEED,
        "epochs": EPOCHS,
        "batch": BATCH,
        "imgsz": IMGSZ,
        "device": DEVICE,
        "patience": PATIENCE,
        "workers": WORKERS,
        "save_period": SAVE_PERIOD,
        "fraction": 1.0,
        "hardware": evaluation_hardware,
        "pretrained_sha256": pretrained_sha256,
        "dataset": dataset_identities[variant],
        "versions": {p: md.version(p) for p in ["ultralytics", "torch", "torchvision"]},
    }
    config_path = directory / "study.json"
    if config_path.exists():
        if read_json(config_path) != identity:
            raise ValueError(f"Experiment settings/dataset changed: {directory}")
    elif directory.exists() and any(directory.iterdir()):
        raise ValueError(f"Unknown existing experiment: {directory}")
    else:
        write_json(config_path, identity)
    best, last = directory / "weights/best.pt", directory / "weights/last.pt"
    assert last.exists() or not (directory / "results.csv").exists(), (
        "Training history exists without last.pt; recover the checkpoint or use a fresh root"
    )
    completed = directory / "completed.json"
    if completed.exists():
        hashes = read_json(completed)
        assert hashes["study_identity"] == digest(identity)
        assert file_hash(best) == hashes["best_sha256"] and file_hash(last) == hashes["last_sha256"]
        status = "verified completed reuse"
    else:
        finished = last.exists() and YOLO(str(last)).ckpt["epoch"] == -1
        if not finished:
            if last.exists() and not RESUME:
                raise ValueError(
                    "Interrupted checkpoint exists; enable RESUME or choose a fresh root"
                )
            model = YOLO(str(last if last.exists() else pretrained))

            def show_progress(trainer, variant=variant):
                history_path = trainer.save_dir / "results.csv"
                if not history_path.is_file():
                    return
                history = pd.read_csv(history_path).rename(columns=str.strip)
                clear_output(wait=True)
                print(f"{variant}: {len(history)}/{EPOCHS} epochs; train-derived tune only")
                display(history.tail(1))
                fig, axes = plt.subplots(1, 3, figsize=(11, 3))
                for key in ["train/box_loss", "val/box_loss"]:
                    axes[0].plot(history.epoch, history[key], label=key)
                axes[0].set_title("Box loss")
                for key in ["metrics/mAP50(B)", "metrics/mAP50-95(B)"]:
                    axes[1].plot(history.epoch, history[key], label=key)
                axes[1].set_title("Tune AP (native)")
                axes[2].plot(history.epoch, history.time / 60)
                axes[2].set_title("Cumulative training minutes")
                for ax in axes:
                    ax.set_xlabel("Epoch")
                axes[0].legend(fontsize=7)
                axes[1].legend(fontsize=7)
                fig.tight_layout()
                plt.show()

            model.add_callback("on_fit_epoch_end", show_progress)
            if last.exists():
                model.train(resume=True)
                status = "native resume"
            else:
                model.train(
                    data=str(DATASETS[variant] / "data.yaml"),
                    fraction=1.0,
                    epochs=EPOCHS,
                    batch=BATCH,
                    imgsz=IMGSZ,
                    device=DEVICE,
                    seed=SEED,
                    deterministic=True,
                    workers=WORKERS,
                    patience=PATIENCE,
                    save=True,
                    save_period=SAVE_PERIOD,
                    project=str(EXPERIMENT_ROOT),
                    name=variant,
                    exist_ok=True,
                    plots=True,
                )
                status = "native fresh training"
        else:
            status = "completed native artifacts recovered"
        assert best.is_file() and last.is_file()
        # Native final_eval strips optimizer and marks epoch=-1, including early stopping.
        assert YOLO(str(last)).ckpt["epoch"] == -1
        assert 0 < len(pd.read_csv(directory / "results.csv")) <= EPOCHS
        write_json(
            completed,
            {
                "best_sha256": file_hash(best),
                "last_sha256": file_hash(last),
                "study_identity": digest(identity),
            },
        )
    checkpoints[variant] = best
    training_log.append(
        {
            "variant": variant,
            "status": status,
            "best": str(best),
            "last": str(last),
            "epochs": len(pd.read_csv(directory / "results.csv")),
        }
    )
assert file_hash(pretrained) == pretrained_sha256
display(pd.DataFrame(training_log))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(11, 3))
for variant in TRAIN_VARIANTS:
    history = pd.read_csv(EXPERIMENT_ROOT / variant / "results.csv").rename(columns=str.strip)
    for ax, loss in zip(axes, ["box_loss", "cls_loss", "dfl_loss"], strict=True):
        for split, style in [("train", "-"), ("val", "--")]:
            key = f"{split}/{loss}"
            if key in history:
                ax.plot(history.epoch, history[key], style, marker="o", label=f"{variant} {split}")
        ax.set(title=loss, xlabel="Epoch", ylabel="Native loss")
        ax.legend(fontsize=6)
fig.suptitle("Native training / train-derived tuning losses")
fig.tight_layout()
plt.show()

## Freeze settings and evaluate both questions with native `model.val()`

Every comparison below uses the same exported test IDs and byte-identical label
files. `EVAL_SPLIT` is explicit (tune during development); training's `val` remains tune. The raw-original checkpoint
hash is verified before/after the entire matrix. The whole matrix uses identical
native confidence/NMS/image-size/precision settings, without custom metrics.
Per-class AP uses only `ap_class_index`; absent classes are **not** assigned the
overall mAP through `box.maps`.

Timing is Ultralytics' native preprocessing/inference/postprocess speed on files
that are **already processed offline**. It excludes offline preparation and is
not measured online end-to-end latency or application throughput.

Completed native evaluation pairs are reused only when selection, verified dataset/encoding identities, weight hashes, val settings, software and hardware match. Cached metrics and native plots are checksummed. Set `FORCE_REEVALUATE=True` to intentionally measure again; cached speeds are the **original measurement**, not fresh rerun timings. No per-prediction shards or custom metrics are introduced. Startup uses manifest/inventory checks only; `DEEP_VERIFY_EXPORTS=True` requests on-demand image hashing/decoding.

In [ ]:
assert EVAL_SPLIT != "test" or FROZEN_FOR_TEST
frozen_checkpoints = {v: file_hash(p) for v, p in checkpoints.items()}
evaluation_identity = {
    "selection": selection["selection_id"],
    "split": EVAL_SPLIT,
    "datasets": dataset_identities,
    "weights": frozen_checkpoints,
    "settings": VAL_SETTINGS,
    "versions": evaluation_versions,
    "hardware": evaluation_hardware,
    "train_variants": TRAIN_VARIANTS,
    "eval_variants": EVAL_VARIANTS,
}
identity_path = EXPERIMENT_ROOT / "evaluation.json"
if (
    identity_path.exists()
    and read_json(identity_path) != evaluation_identity
    and not FORCE_REEVALUATE
):
    raise ValueError("Evaluation identity changed; choose a new root or FORCE_REEVALUATE=True")
write_json(identity_path, evaluation_identity)
rows, class_rows, evaluation_log = [], [], []
for trained, best in checkpoints.items():
    for evaluated in EVAL_VARIANTS:
        output = EXPERIMENT_ROOT / "evaluation" / f"{trained}_on_{evaluated}_{EVAL_SPLIT}"
        pair_identity = {
            "selection": selection["selection_id"],
            "split": EVAL_SPLIT,
            "trained_on": trained,
            "evaluated_on": evaluated,
            "training_dataset": dataset_identities[trained],
            "evaluation_dataset": dataset_identities[evaluated],
            "weights_sha256": frozen_checkpoints[trained],
            "settings": VAL_SETTINGS,
            "versions": evaluation_versions,
            "hardware": evaluation_hardware,
        }
        cache_path = output / "native_result.json"
        cached = reuse_evaluation(cache_path, pair_identity, force=FORCE_REEVALUATE)
        if cached is not None:
            row, pair_classes = cached["overall"], cached["classes"]
            reused = True
        else:
            model = YOLO(str(best))
            runtime = {}
            model.add_callback(
                "on_val_start",
                lambda validator, runtime=runtime: runtime.update(device=str(validator.device)),
            )
            metrics = model.val(
                data=str(DATASETS[evaluated] / "data.yaml"),
                split="val" if EVAL_SPLIT == "tune" else "test",
                project=str(output.parent),
                name=output.name,
                exist_ok=True,
                **VAL_SETTINGS,
            )
            assert model.names == dict(enumerate(CLASSES))
            assert runtime["device"] == (f"cuda:{DEVICE}" if DEVICE.isdigit() else DEVICE)
            values = [metrics.box.map50, metrics.box.map, metrics.box.mp, metrics.box.mr]
            assert np.isfinite(values).all() and all(0 <= x <= 1 for x in values)
            row = {
                "trained_on": trained,
                "evaluated_on": evaluated,
                "split": EVAL_SPLIT,
                "images": len(selection["splits"][EVAL_SPLIT]),
                "mAP50": float(metrics.box.map50),
                "mAP50_95": float(metrics.box.map),
                "precision": float(metrics.box.mp),
                "recall": float(metrics.box.mr),
                "checkpoint_sha256": frozen_checkpoints[trained],
                "actual_device": runtime["device"],
                **{f"{k}_ms": float(v) for k, v in metrics.speed.items()},
            }
            row["native_stage_ms"] = sum(metrics.speed.values())
            pair_classes = [
                {
                    "trained_on": trained,
                    "evaluated_on": evaluated,
                    "class": CLASSES[int(cls)],
                    "AP50": float(metrics.box.ap50[j]),
                    "AP50_95": float(metrics.box.ap[j]),
                }
                for j, cls in enumerate(metrics.box.ap_class_index)
            ]
            plots = [
                output / "confusion_matrix_normalized.png",
                *sorted(output.glob("*PR_curve.png")),
            ]
            assert len(plots) >= 2 and all(path.is_file() for path in plots)
            payload = {
                "overall": row,
                "classes": pair_classes,
                "artifacts": {path.name: file_hash(path) for path in plots},
            }
            write_json(
                cache_path,
                {"identity": pair_identity, "result": payload, "checksum": digest(payload)},
            )
            reused = False
        rows.append(row)
        class_rows.extend(pair_classes)
        evaluation_log.append({"trained_on": trained, "evaluated_on": evaluated, "reused": reused})
assert all(file_hash(checkpoints[v]) == sha for v, sha in frozen_checkpoints.items())
results, per_class = pd.DataFrame(rows), pd.DataFrame(class_rows)
results.to_csv(EXPERIMENT_ROOT / "results.csv", index=False)
write_json(EXPERIMENT_ROOT / "results.json", results.to_dict("records"))
per_class.to_csv(EXPERIMENT_ROOT / "per_class_AP.csv", index=False)
display(pd.DataFrame(evaluation_log))
display(results.drop(columns=["checkpoint_sha256"]).round(5))
display(
    per_class.pivot(index=["trained_on", "evaluated_on"], columns="class", values="AP50_95")
    .reindex(columns=CLASSES)
    .round(5)
)

In [ ]:
frozen_raw = results[results.trained_on == "raw_original"].set_index("evaluated_on")
control = "raw_reencoded" if "raw_reencoded" in EVAL_VARIANTS else "raw_original"
common_test = results[results.evaluated_on == control].set_index("trained_on")
frozen_raw["delta_vs_control"] = frozen_raw.mAP50_95 - frozen_raw.loc[control, "mAP50_95"]
common_test["delta_vs_control_training"] = (
    common_test.mAP50_95
    - common_test.loc[
        "raw_reencoded" if "raw_reencoded" in TRAIN_VARIANTS else "raw_original", "mAP50_95"
    ]
)
print(f"Frozen raw checkpoint · {EVAL_SPLIT} inputs · delta control: {control}")
display(
    frozen_raw[["mAP50", "mAP50_95", "delta_vs_control", "inference_ms", "native_stage_ms"]].round(
        5
    )
)
print(f"Trained checkpoints on common {control} files · {EVAL_SPLIT}")
display(
    common_test[
        ["mAP50", "mAP50_95", "delta_vs_control_training", "inference_ms", "native_stage_ms"]
    ].round(5)
)
fig, axes = plt.subplots(1, 3, figsize=(11, 3))
results.pivot(index="trained_on", columns="evaluated_on", values="mAP50_95").plot.bar(
    ax=axes[0], title="Native mAP50:95"
)
results.assign(pair=results.trained_on + "→" + results.evaluated_on).set_index(
    "pair"
).inference_ms.plot.bar(ax=axes[1], title="Native inference only (ms)")
preparation = pd.read_csv(RUN_ROOT / "preparation_cost.csv").set_index("variant")
(preparation.preparation_ms / 1000).plot.bar(ax=axes[2], title="Separate offline preparation (s)")
for ax in axes:
    ax.tick_params(axis="x", rotation=40)
fig.tight_layout()
plt.show()
per_class[per_class.trained_on == "raw_original"].pivot(
    index="class", columns="evaluated_on", values="AP50_95"
).reindex(CLASSES).plot.bar(figsize=(8, 3), title=f"Native per-class AP50:95 · {EVAL_SPLIT}")
plt.tight_layout()
plt.show()
metadata = pd.read_csv(RUN_ROOT / "metadata.csv")
costs = metadata.groupby("variant")[["preprocess_ms", "preparation_ms"]].agg(
    ["mean", "median", lambda x: x.quantile(0.95)]
)
display(costs.rename(columns={"<lambda_0>": "p95"}))

In [ ]:
def show_native_plot(path):
    with PILImage.open(path) as im:
        im.thumbnail((800, 500))
        buffer = io.BytesIO()
        im.save(buffer, format="PNG", optimize=True)
    display(Image(data=buffer.getvalue()))


baseline_plots = EXPERIMENT_ROOT / f"evaluation/raw_original_on_raw_original_{EVAL_SPLIT}"
for pattern in ["confusion_matrix_normalized.png", "*PR_curve.png"]:
    paths = sorted(baseline_plots.glob(pattern))
    assert paths, f"Native plot missing: {pattern}"
    print(paths[0].name)
    show_native_plot(paths[0])
print(
    "All matrix plots, labels/prediction grids and curves remain under each evaluation directory."
)

## Inspect diagnostic failure examples — no handwritten matching/AP

The following examples rank absolute **GT vs predicted object-count differences**
at display confidence 0.25. This is only a cheap diagnostic: equal counts do not
imply correct localization/classes, and it is not AP or an accuracy ranking.
Native confusion matrices and PR curves above provide the quantitative analysis.
Native predictions below read exported raw-original evaluation files; no custom filtering occurs.

This bounded diagnostic inspects the first 32 selected evaluation examples, not
the complete dataset. It does not support a representative failure-rate claim.

In [ ]:
meta = pd.read_csv(DATASETS["raw_original"] / "metadata.csv")
# shortcut: inspect 32 examples; expand for final failure analysis.
test_rows = meta[meta.split == EVAL_SPLIT].head(32).to_dict("records")
fig, axes = plt.subplots(
    len(TRAIN_VARIANTS), 2, figsize=(8, 3 * len(TRAIN_VARIANTS)), squeeze=False
)
failure_rows = []
for i, variant in enumerate(TRAIN_VARIANTS):
    candidates = []
    predictor = YOLO(str(checkpoints[variant]))
    for r, pred in zip(
        test_rows,
        predictor.predict(
            source=[r["image"] for r in test_rows],
            stream=True,
            device=DEVICE,
            imgsz=IMGSZ,
            conf=0.25,
            quantize=32,
            verbose=False,
        ),
        strict=True,
    ):
        label = DATASETS["raw_original"] / f"labels/{EVAL_SPLIT}" / f"{r['id']}.txt"
        text = label.read_text().strip()
        gt = np.loadtxt(label, ndmin=2).reshape(-1, 5) if text else np.empty((0, 5))
        candidates.append((abs(len(gt) - len(pred.boxes)), r, pred, gt))
    gap, r, pred, gt = max(candidates, key=lambda item: (item[0], item[1]["id"]))
    image = read_image(r["image"])
    h, w = image.shape[:2]
    for box in xywh2xyxy(gt[:, 1:] * np.array([w, h, w, h])):
        x1, y1, x2, y2 = map(round, box)
        cv2.rectangle(image, (x1, y1), (x2, y2), (0, 255, 0), 2)
    axes[i, 0].imshow(cv2.cvtColor(image, cv2.COLOR_BGR2RGB))
    axes[i, 0].set_title(f"{variant} common raw {EVAL_SPLIT} · GT {len(gt)}")
    axes[i, 1].imshow(cv2.cvtColor(pred.plot(), cv2.COLOR_BGR2RGB))
    axes[i, 1].set_title(f"Native detections {len(pred.boxes)} · count gap {gap}")
    failure_rows.append(
        {
            "trained_on": variant,
            "id": r["id"],
            "GT_count": len(gt),
            "prediction_count": len(pred.boxes),
            "count_gap": gap,
        }
    )
for ax in axes.flat:
    ax.axis("off")
fig.tight_layout()
plt.show()
display(pd.DataFrame(failure_rows))

In [ ]:
# Check completed-run reuse without another optimizer step.
for variant in TRAIN_VARIANTS:
    directory = EXPERIMENT_ROOT / variant
    saved = read_json(directory / "completed.json")
    assert file_hash(checkpoints[variant]) == saved["best_sha256"]
    assert file_hash(directory / "weights/last.pt") == saved["last_sha256"]
assert verify_alignment(DATASETS.values(), deep=False) == sum(
    len(v) for v in selection["splits"].values()
)
validation = {
    "mode": "real BDD offline export + separately trained YOLO + native model.val, no mocks",
    "evaluation_split": EVAL_SPLIT,
    "holdout_evaluated": EVAL_SPLIT == "test",
    "patience": PATIENCE,
    "epochs_completed": {
        v: len(pd.read_csv(EXPERIMENT_ROOT / v / "results.csv")) for v in TRAIN_VARIANTS
    },
    "selection_id": selection["selection_id"],
    "split_sizes": {s: len(v) for s, v in selection["splits"].items()},
    "epochs": EPOCHS,
    "model": MODEL,
    "seed": SEED,
    "device": DEVICE,
    "checkpoint_sha256": frozen_checkpoints,
    "evaluated_pairs": len(results),
    "evaluation_pairs_reused": sum(item["reused"] for item in evaluation_log),
    "dataset_encodings": {v: dataset_identities[v]["export"]["encoding"] for v in VARIANTS},
    "native_metrics": True,
    "online_filtering": False,
    "versions": {p: md.version(p) for p in ["bdd100k", "scalabel", "ultralytics", "torch"]},
}
write_json(EXPERIMENT_ROOT / "validation.json", validation)
display(validation)
print("PASS: frozen models, aligned native matrix and completed checkpoint reuse")

## Phase 2 interpretation and remaining work

The full raw baseline is the first gate. Tune metrics support model/recipe decisions;
they are not final holdout estimates. Stop before final test until train/tune-only
calibration, the JPEG95 control and all requested fixed/adaptive experiments have
been finalized and their identities frozen. Use the same raw-trained checkpoint,
split IDs, label bytes and evaluation settings across input treatments.

The training cell refreshes actual native curves after each epoch. Native validation
plots include confusion and PR; per-class AP and exported-file model latency appear
after training completes. Offline costs remain separate from model stage latency.
Native average speed is not a synchronized batch-1 p50/p95 deployment benchmark.

Calibration, all treatment measurements, robustness, repeated seeds/uncertainty and
final holdout release remain tracked in `docs/PHASES.md` and Issue #1. Missing
experiments have no scores. No pilot or mock result substitutes for full training.